# SLURM scripts (Altamira)

In [8]:
print("Minutes per model trainig-testing for Job duration") # 20-lhs x 10-sp = 200 runs per job
print([(h,h*60/200) for h in [6,8,10,12,15,18,20]])

Minutes per model trainig-testing for Job duration
[(6, 1.8), (8, 2.4), (10, 3.0), (12, 3.6), (15, 4.5), (18, 5.4), (20, 6.0)]


In [14]:
# Create SLURM scripts for running a set of jobs at Altamira cloud computing
# Random Forest, XGBoost, Support Vector Machine, Elastic Net, EBM

n_trial=0 # LHS trial number !!!!!!!!!

for model in ['rf','xgb','sv','en','ebm','fastai']:
    # time limit for each model
    if model in ['xgb','sv','en']:
        time="6:00:00"
    elif model in ['fastai']:
        time="10:00:00"
    elif model in ['rf','ebm']:
        time="20:00:00"
    for sp in range(9):
        with open(f"slurm_lhs{n_trial}_model_selection_{model}_sp{sp}.sh", "w") as f:
            f.write("#!/bin/bash\n")
            f.write(f"#SBATCH --job-name=lhs{n_trial}_msel_{model}_{sp}"+"\n")
            f.write(f"#SBATCH --output=lhs{n_trial}_msel_{model}_{sp}.out"+"\n")
            f.write(f"#SBATCH --error=lhs{n_trial}_msel_{model}_{sp}.err"+"\n")
            f.write("#SBATCH --time="+time+"\n")
            f.write("#SBATCH --ntasks=1\n") # 1 tasks per job
            f.write("#SBATCH --cpus-per-task=16\n")
            f.write("#SBATCH --chdir=/home/uc27/uc27001/glassboxevm/\n")
            f.write('module purge\n')
            f.write('module load PYTHON\n')
            f.write('source $HOME/mlenv/bin/activate\n')
            f.write(f"srun -n 1 --ntasks-per-node=16 python3 model_selection.py {model} {n_trial} {sp}")  
        f.close()
